# WAV1 paired multiseed confirmation — Kaggle
Attach exactly the private `faruq-v3-experiment-core-v1` dataset and the Saved Version output from the completed standalone WAV1 seed-42 notebook. This run reuses seed 42 and trains only WAV1 seeds 123 and 2026. Locked test remains closed.


In [ ]:
from pathlib import Path
import hashlib
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
if not INPUT.is_dir() or not WORK.is_dir(): raise RuntimeError('Notebook ini Kaggle-only.')
manifests=sorted(p for p in INPUT.rglob('af2_spectral_kaggle_manifest.json') if p.is_file())
if len(manifests)!=1: raise FileNotFoundError(f'STOP CEPAT: core manifest={manifests}')
for name in ('D0_seed123_best.pt','D0_seed2026_best.pt','af2_igem_paired_confirmation.json'):
    hits=sorted(p for p in INPUT.rglob(name) if p.is_file())
    if len(hits)!=1: raise FileNotFoundError(f'STOP CEPAT: {name}={hits}')
wav_hits=sorted(p for p in INPUT.rglob('WAV1_seed42_result.json') if p.is_file())
groups={}
for p in wav_hits:
    h=hashlib.sha256(p.read_bytes()).hexdigest(); groups.setdefault(h,[]).append(p)
if not groups: raise FileNotFoundError('Attach Saved Version output WAV1 seed42.')
if len(groups)!=1: raise RuntimeError(f'WAV1 seed42 ambigu/non-identik: {wav_hits}')
WAV42=sorted(next(iter(groups.values())))[0]
print('FAST INPUT PREFLIGHT PASS'); print('CORE:',manifests[0]); print('WAV1 SEED42:',WAV42)


In [ ]:
import importlib,json,os,shutil,subprocess,sys,time,hashlib
from pathlib import Path
WORK=Path('/kaggle/working'); INPUT=Path('/kaggle/input'); REPO=WORK/'coffee-bean-detection'; OUT=WORK/'wav1-paired-confirmation-v1'; BRANCH='agent/wav1-paired-confirmation'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(3):
    r=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
for m in list(sys.modules):
    if m=='coffee_detector' or m.startswith('coffee_detector.'): sys.modules.pop(m,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip(); print('COMMIT:',COMMIT)
from coffee_detector.experiments.prepare_af2_spectral_kaggle import prepare_af2_spectral_kaggle_input
from coffee_detector.experiments.run_faruq_v3_wav1_paired_confirmation import CONFIG
DATA,A,CONTRACT=prepare_af2_spectral_kaggle_input(INPUT,WORK); assert CONTRACT['decision']=='PASS' and CONTRACT['test_images_accessed'] is False
D123=A['D0_seed123_best.pt']; D2026=A['D0_seed2026_best.pt']; REF=A['af2_igem_paired_confirmation.json']
OUT.mkdir(exist_ok=True); (OUT/'val_reports').mkdir(parents=True,exist_ok=True)
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def restore(seed,d0):
    expected={'format':'coffee_detector.wav1_paired.run_contract.v1','arm':'WAV1','seed':seed,'config_sha256':sha(CONFIG),'d0_checkpoint_sha256':sha(d0),'epochs':50,'test_images_accessed':False}
    matches=[]
    for cp in INPUT.rglob('run_contract.json'):
        try: payload=json.loads(cp.read_text(encoding='utf-8'))
        except Exception: continue
        if payload==expected: matches.append(cp.parent)
    if not matches: return None
    if len(matches)!=1: raise RuntimeError(f'Resume WAV1 seed {seed} ambigu: {matches}')
    dst=OUT/'WAV1'/f'WAV1_seed{seed}'
    if not dst.exists(): dst.parent.mkdir(parents=True,exist_ok=True); shutil.copytree(matches[0],dst)
    return dst
print('RESTORE 123:',restore(123,D123)); print('RESTORE 2026:',restore(2026,D2026))
LOG=OUT/'wav1_paired_confirmation.log'; RESULT=OUT/'val_reports/wav1_paired_confirmation.json'
CMD=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_wav1_paired_confirmation','--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),'--wav1-seed42-result',str(WAV42),'--af2-igem-reference',str(REF),'--d0-seed123',str(D123),'--d0-seed2026',str(D2026),'--output-root',str(OUT),'--device','0','--authorize-training']
if not RESULT.is_file():
    with LOG.open('a',encoding='utf-8') as stream: process=subprocess.Popen(CMD,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
    seen={}
    while process.poll() is None:
        for seed in (123,2026):
            csv=OUT/'WAV1'/f'WAV1_seed{seed}'/'results.csv'; n=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
            if seen.get(seed)!=n: print(f'WAV1 seed {seed}: {n}/50 epoch | log={LOG}',flush=True); seen[seed]=n
        time.sleep(120)
    if process.returncode:
        tail='\n'.join(LOG.read_text(errors='replace').splitlines()[-220:]) if LOG.is_file() else '<log tidak ditemukan>'
        raise RuntimeError(f'WAV1 paired gagal: returncode={process.returncode}\n--- LOG TAIL ---\n{tail}')
if not RESULT.is_file(): raise FileNotFoundError(RESULT)
result=json.loads(RESULT.read_text(encoding='utf-8')); assert result['test_opened'] is False and result['test_images_accessed'] is False
print('=== WAV1 THREE-SEED DECISION ==='); print(json.dumps(result,indent=2))
HANDOFF=WORK/'wav1-paired-confirmation-handoff'; shutil.rmtree(HANDOFF,ignore_errors=True); (HANDOFF/'val_reports').mkdir(parents=True)
for p in (OUT/'val_reports').glob('*.json'): shutil.copy2(p,HANDOFF/'val_reports'/p.name)
if result['decision']=='PASS':
    for seed in (123,2026):
        best=OUT/'WAV1'/f'WAV1_seed{seed}'/'weights/best.pt'
        if best.is_file(): shutil.copy2(best,HANDOFF/f'WAV1_seed{seed}_best.pt')
manifest={'format':'coffee_detector.wav1_paired.handoff.v1','commit':COMMIT,'seeds':[42,123,2026],'decision':result['decision'],'test_images_accessed':False}
(HANDOFF/'handoff_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n',encoding='utf-8')
if DATA.exists(): shutil.rmtree(DATA)
if result['decision'] in ('PASS','FAIL'): shutil.rmtree(OUT/'WAV1',ignore_errors=True)
if REPO.exists(): shutil.rmtree(REPO)
print('HANDOFF READY:',HANDOFF); print('SAVE VERSION. Tidak perlu download checkpoint.')
